# AI-Powered Paraphrasing Tool

## Objective

The objective of this project is to develop an **AI-powered paraphrasing tool using Python and deep learning**. The tool uses a **Transformer-based T5 model from Hugging Face** to generate a paraphrased version of the given input text.

The system is designed to:

- Accept a block of text from the user.
- Generate a paraphrased version while preserving the original meaning.
- Improve the clarity and readability of the text.
- Check the generated output for grammar and spelling issues.
- Evaluate the similarity between the original and paraphrased text using ROUGE scores.
- Provide a simple evaluation report for the generated output.

The application is implemented as a **console/script-based program without a GUI or web interface**.

In [1]:
!pip -q install transformers sentencepiece language-tool-python nltk rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 930.3 kB/s eta 0:00:00


In [2]:
import re
import nltk
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
import language_tool_python
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Model loaded successfully!")
print("Using device:", device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Model loaded successfully!
Using device: cpu


In [4]:
def paraphrase_text(text):
    input_text = "paraphrase: " + text + " </s>"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}
    output = model.generate(
        **inputs,
        max_length=512,
        num_beams=5,
        num_return_sequences=1,
        early_stopping=True
    )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
    return result

In [5]:
tool = language_tool_python.LanguageTool("en-US")
def check_grammar(text):
    matches = tool.check(text)
    print("Grammar and spelling check")
    print("-" * 40)
    if len(matches) == 0:
        print("No grammar or spelling problems found.")
    else:
        print("Number of possible issues:", len(matches))
        for match in matches[:10]:
            print("Problem:", match.message)
            print("Suggestion:", ", ".join(match.replacements[:3]))
            print()
    return matches

In [6]:
def similarity_check(original, paraphrased):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"],
        use_stemmer=True
    )
    scores = scorer.score(original, paraphrased)
    print("Similarity Check")
    print("-" * 40)
    print("ROUGE-1:", round(scores["rouge1"].fmeasure, 4))
    print("ROUGE-L:", round(scores["rougeL"].fmeasure, 4))
    return scores

In [7]:
print("=" * 60)
print("AI POWERED PARAPHRASING TOOL")
print("=" * 60)
text = input("\nEnter the text you want to paraphrase:\n")
if text.strip() == "":
    print("Please enter some text.")
else:
    print("\nGenerating paraphrase...")
    paraphrased_text = paraphrase_text(text)
    print("\nOriginal Text:")
    print(text)
    print("\nParaphrased Text:")
    print(paraphrased_text)
    print("\n")
    check_grammar(paraphrased_text)
    print("\n")
    similarity_check(text, paraphrased_text)

AI POWERED PARAPHRASING TOOL

Enter the text you want to paraphrase:
keralam

Generating paraphrase...

Original Text:
keralam

Paraphrased Text:
Keralam:


Grammar and spelling check
----------------------------------------
Number of possible issues: 1
Problem: Possible spelling mistake found.
Suggestion: Kerala



Similarity Check
----------------------------------------
ROUGE-1: 1.0
ROUGE-L: 1.0


In [10]:
sample_text = """
Machine learning is a branch of artificial intelligence that allows
computers to learn from data and make predictions without being
explicitly programmed for every task.
"""
print("Original Text:")
print(sample_text)
sample_result = paraphrase_text(sample_text)
print("\nParaphrased Text:")
print(sample_result)
print("\nGrammar Check:")
check_grammar(sample_result)
print("\nEvaluation:")
similarity_check(sample_text, sample_result)

Original Text:

Machine learning is a branch of artificial intelligence that allows
computers to learn from data and make predictions without being
explicitly programmed for every task.


Paraphrased Text:
Machine learning is a branch of artificial intelligence that allows computers to learn from data and make predictions without being explicitly programmed for every task .

Grammar Check:
Grammar and spelling check
----------------------------------------
Number of possible issues: 1
Problem: Don’t put a space before the full stop.
Suggestion: .


Evaluation:
Similarity Check
----------------------------------------
ROUGE-1: 1.0
ROUGE-L: 1.0


{'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0)}

In [11]:
def evaluation_report(original, paraphrased):
    original_words = len(original.split())
    new_words = len(paraphrased.split())
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"],
        use_stemmer=True
    )
    scores = scorer.score(original, paraphrased)
    print("=" * 50)
    print("EVALUATION REPORT")
    print("=" * 50)
    print("Original word count   :", original_words)
    print("Paraphrased word count:", new_words)
    print("ROUGE-1 score         :", round(
        scores["rouge1"].fmeasure, 4
    ))
    print("ROUGE-L score         :", round(
        scores["rougeL"].fmeasure, 4
    ))
    if scores["rougeL"].fmeasure > 0.4:
        print("Meaning preservation  : Reasonable")
    else:
        print("Meaning preservation  : Needs review")

    print("\nThe paraphrased text should also be manually")
    print("checked for meaning, grammar and readability.")

In [12]:
evaluation_report(sample_text, sample_result)

EVALUATION REPORT
Original word count   : 25
Paraphrased word count: 26
ROUGE-1 score         : 1.0
ROUGE-L score         : 1.0
Meaning preservation  : Reasonable

The paraphrased text should also be manually
checked for meaning, grammar and readability.


In [13]:
print("=" * 60)
print("FINAL PARAPHRASING EVALUATION")
print("=" * 60)
print("\nOriginal Text:")
print(sample_text)
print("\nParaphrased Text:")
print(sample_result)
print("\nGrammar and Spelling:")
matches = check_grammar(sample_result)
print("\nSimilarity Scores:")
scores = similarity_check(sample_text, sample_result)
print("\nFinal Assessment:")
if len(matches) == 0:
    print("Grammar and spelling: Good")
else:
    print("Grammar and spelling: Some issues found")

if scores["rougeL"].fmeasure >= 0.4:
    print("Meaning preservation: Reasonable")
else:
    print("Meaning preservation: Needs manual review")

print("Fluency: The generated sentence should be manually checked")
print("for natural wording, readability and clarity.")

print("\nOverall: The paraphrased text should be reviewed")
print("before being considered as the final output.")

FINAL PARAPHRASING EVALUATION

Original Text:

Machine learning is a branch of artificial intelligence that allows
computers to learn from data and make predictions without being
explicitly programmed for every task.


Paraphrased Text:
Machine learning is a branch of artificial intelligence that allows computers to learn from data and make predictions without being explicitly programmed for every task .

Grammar and Spelling:
Grammar and spelling check
----------------------------------------
Number of possible issues: 1
Problem: Don’t put a space before the full stop.
Suggestion: .


Similarity Scores:
Similarity Check
----------------------------------------
ROUGE-1: 1.0
ROUGE-L: 1.0

Final Assessment:
Grammar and spelling: Some issues found
Meaning preservation: Reasonable
Fluency: The generated sentence should be manually checked
for natural wording, readability and clarity.

Overall: The paraphrased text should be reviewed
before being considered as the final output.
